# Starter notebook: multimodal representations of a CRISPR screen

This notebook loads the tables of the Element Biosciences Cytos screen, where RNA, protein and cell painting were measured in the same 258,896 A549 cells. It builds one embedding from all three modalities. It runs on a laptop in a few minutes and needs about 4 GB of memory.

The README describes the experimental design and every column.

In [ ]:
import urllib.request
import zipfile
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

DATA = Path("data")
URL = "https://exampledata.scverse.org/sbi2-cytodata/multiomics"
FILES = ["rna.h5ad", "imaging.h5ad", "guides.h5ad"]  # 1.5 GB together

DATA.mkdir(exist_ok=True)
for name in FILES:
    if not (DATA / name).exists():
        urllib.request.urlretrieve(f"{URL}/{name}", DATA / name)

## Load the tables

The three files share the same rows in the same order, so row `i` is the same cell in every file. We open the files without loading their matrices, keep the cells that pass QC (`qc_pass`, which combines all QC flags), and load a random sample of 20,000 of them, which keeps memory and runtime low.

In [ ]:
rna_all = ad.read_h5ad(DATA / "rna.h5ad", backed="r")
img_all = ad.read_h5ad(DATA / "imaging.h5ad", backed="r")
assert (rna_all.obs_names == img_all.obs_names).all()

obs = rna_all.obs
pd.crosstab(obs["target_gene"], [obs["condition"], obs["qc_pass"]])

In [ ]:
rng = np.random.default_rng(0)
cells = np.sort(rng.choice(np.flatnonzero(obs["qc_pass"]), 20_000, replace=False))
rna = rna_all[cells].to_memory()
img = img_all[cells].to_memory()
obs = rna.obs

## Split the imaging table into protein and cell painting

Each imaging feature has a `modality` in `var`: `cell_paint`, `protein`, `shape`, `control` (IgG) or `background`. The imaging matrix was normalised with mantispy: each feature is a robust z-score against all non-targeting cells, and `var["selected"]` marks the features that mantispy's feature selection kept. The raw values are in `layers["raw"]`. We use the selected protein features as one block, and the selected cell painting and shape features as another.

In [ ]:
selected, modality = img.var["selected"], img.var["modality"]
protein = img[:, selected & (modality == "protein")].to_df()
paint = img[:, selected & modality.isin(["cell_paint", "shape"])].to_df()
print(f"protein: {protein.shape[1]} features, cell painting: {paint.shape[1]} features")

## One PCA per modality

For RNA we normalise the counts, log-transform them and keep 2,000 highly variable genes. The image features are already normalised, so we only fill the few missing values and clip extreme values. Each modality then gets its own PCA.

In [ ]:
sc.pp.normalize_total(rna, target_sum=1e4)
sc.pp.log1p(rna)
sc.pp.highly_variable_genes(rna, n_top_genes=2000)
rna_hvg = rna[:, rna.var["highly_variable"]].X.toarray()


def clean(df):
    return df.fillna(0).clip(-10, 10).to_numpy()


blocks = {
    "rna": StandardScaler().fit_transform(rna_hvg),
    "protein": clean(protein),
    "paint": clean(paint),
}
n_pcs = {"rna": 30, "protein": 10, "paint": 30}
pcs = {k: PCA(n_pcs[k], random_state=0).fit_transform(x) for k, x in blocks.items()}

## Joint embedding

We scale each block of principal components so that each modality has the same total variance, and then concatenate the blocks. This step keeps any single modality from dominating the embedding just because it has more features. We then compute one UMAP for the joint embedding and one for each modality, so you can compare them.

In [ ]:
joint = np.hstack([p / np.sqrt((p**2).sum(1).mean()) for p in pcs.values()])

emb = sc.AnnData(obs=obs[["Well", "condition", "target_gene", "guide"]].copy())
for name, x in {**pcs, "joint": joint}.items():
    emb.obsm[f"X_{name}"] = x
    sc.pp.neighbors(emb, use_rep=f"X_{name}", key_added=name)
    emb.obsm[f"X_umap_{name}"] = sc.tl.umap(emb, neighbors_key=name, copy=True).obsm["X_umap"]

In [ ]:
sc.pl.embedding(emb, "X_umap_joint", color=["condition", "target_gene", "Well"], ncols=3)

In [ ]:
for name in ["rna", "protein", "paint"]:
    sc.pl.embedding(emb, f"X_umap_{name}", color=["condition", "target_gene"], title=[f"{name}: condition", f"{name}: target_gene"])

## Normalising per well

Each well is either untreated or treated with TNFα, so the condition is confounded with the well. The imaging matrix is therefore normalised against the non-targeting cells of the whole plate, which keeps the TNFα response. If you study knockout effects within one condition, you can instead normalise each well against its own non-targeting cells, which also removes differences in staining between wells, with [mantispy](https://github.com/scverse/mantispy):

```python
import mantispy as mt

per_well = img.copy()
per_well.X = per_well.layers["raw"].copy()
mt.pp.normalize(per_well, by="Metadata_Well", reference="negcon")
```

## A known perturbation in all modalities: TGFBR2 knockout

Knocking out TGFBR2 should lower the expression of TGF-β target genes. We compare TGFBR2 knockout cells with non-targeting (NTC) cells in each well, using all QC-passing cells, and we sum the raw counts per well (pseudobulk) so that we don't need to normalise the full matrix.

In [ ]:
genes = ["TGFBR2", "PMEPA1", "TGFBI", "CDH2", "IGFBP7"]
keep = np.flatnonzero(rna_all.obs["qc_pass"].values & rna_all.obs["target_gene"].isin(["TGFBR2", "NTC"]).values)
cols = rna_all.var_names.get_indexer(genes)
# read 5,000 cells at a time and keep only these genes, so memory stays low
x = np.vstack([rna_all.X[keep[i : i + 5000]][:, cols].toarray() for i in range(0, len(keep), 5000)])
sub = rna_all.obs.iloc[keep]
groups = [sub["Well"].values, sub["target_gene"].astype(str).values]
counts = pd.DataFrame(x, columns=genes).groupby(groups).sum()
depth = pd.Series(sub["rna_total_counts"].values).groupby(groups).sum()
cpm = np.log1p(counts.div(depth, axis=0) * 1e6)
log_fc = cpm.xs("TGFBR2", level=1) - cpm.xs("NTC", level=1)
ax = log_fc.plot.box(title="TGFBR2 knockout vs NTC, one point per well (log CPM)")
ax.axhline(0, color="k", lw=0.5);

In [ ]:
feats = {"protein": protein, "cell painting": paint}
rows = []
for modality, df in feats.items():
    z = df.fillna(0)
    diff = z[obs["target_gene"] == "TGFBR2"].mean() - z[obs["target_gene"] == "NTC"].mean()
    rows.append(diff.abs().sort_values(ascending=False).head(5).rename(modality))
pd.concat(rows, axis=1)

## Where to go from here

- Replace the concatenation with a multimodal model, e.g. MOFA or a multimodal autoencoder.
- Check whether the knockouts separate from NTC cells in each modality, and in the joint embedding.
- Predict one modality from another, e.g. RNA from the image features.
- Use the images in `subset_A1_A2.zarr.zip` or `full.zarr.zip` (SpatialData) to learn image features directly.